In [1]:
import pandas as pd

df = pd.read_csv("clean_telemetry.csv")
df.head()

,drone_id,application,drone_size,drone_model,manufacturer,propeller_count,max_carry_weight,actual_carry_weight,payload_type,payload_description,...,battery_remaining,gps_accuracy,wind_speed,obstacles_encountered,flight_status,regulatory_approval_id,notes,overweight_flag,weight_ratio,risk_score
0,D001,Package Delivery,Medium,FlyHigh 300,AeroCorp,4,5.0,2.5,Package,Small consumer goods,...,85,2.1,3.5,No,Completed,REG-2025-ABC-123,"Urban delivery, light package, good weather",0,0.500000,0.897508
1,D002,Infrastructure Inspection,Large,Inspecta X,SkyView Inc.,6,15.0,8.0,"Camera, Sensor","High-resolution camera, thermal sensor",...,68,1.8,6.2,Yes,Completed,REG-2025-DEF-456,"Bridge inspection, carried specialized equipment",0,0.533333,1.356176
2,D003,Agricultural Spraying,Large,CropMaster,AgriDrones,8,20.0,18.0,Liquid Tank,Pesticide solution,...,40,3.5,7.8,Yes,Completed,REG-2025-GHI-789,"Field spraying, full tank, moderate wind",0,0.900000,1.490365
3,D004,Aerial Photography,Small,SnapShot Mini,PhotoFly,4,1.0,0.5,Camera,Lightweight action camera,...,92,1.2,2.1,No,Completed,REG-2025-JKL-012,"Recreational use, clear skies",0,0.500000,1.076124
4,D005,Surveillance,Medium,Watcher Pro,SecureTech,4,7.0,3.0,"Camera, Sensor","Infrared camera, motion detection sensor",...,55,2.5,4.9,No,Completed,REG-2025-MNO-345,"Security patrol, nighttime operation",0,0.428571,1.229845


In [2]:
df.shape
df.info()
df.isna().sum().sort_values(ascending=False).head(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 527 entries, 0 to 526
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   drone_id                527 non-null    object 
 1   application             527 non-null    object 
 2   drone_size              527 non-null    object 
 3   drone_model             527 non-null    object 
 4   manufacturer            527 non-null    object 
 5   propeller_count         527 non-null    int64  
 6   max_carry_weight        527 non-null    float64
 7   actual_carry_weight     527 non-null    float64
 8   payload_type            527 non-null    object 
 9   payload_description     527 non-null    object 
 10  altitude                527 non-null    int64  
 11  flight_duration         527 non-null    float64
 12  distance_flown          527 non-null    float64
 13  operator_id             527 non-null    object 
 14  flight_date             527 non-null    ob

notes                     485
drone_id                    0
operator_id                 0
weight_ratio                0
overweight_flag             0
regulatory_approval_id      0
flight_status               0
obstacles_encountered       0
wind_speed                  0
gps_accuracy                0
battery_remaining           0
flight_date                 0
distance_flown              0
application                 0
flight_duration             0
altitude                    0
payload_description         0
payload_type                0
actual_carry_weight         0
max_carry_weight            0
dtype: int64

In [3]:
# 1. Tái cấu trúc biến mục tiêu từ đa phân loại sang nhị phân (Target Engineering)
df['flight_target'] = df['flight_status'].apply(lambda x: 1 if x == "Completed" else 0)

# 2. Định nghĩa các đặc trưng telemetry dùng cho ML và hệ thống hỗ trợ ra quyết định (DSS)
telemetry_cols = [
    "wind_speed",
    "battery_remaining",
    "actual_carry_weight",
    "payload_type",
    "altitude",
    "distance_flown",
    "gps_accuracy",
    "obstacles_encountered"
]
X = df[telemetry_cols]
y = df['flight_target']

print("Phân phối nhãn mục tiêu nhị phân phục vụ DSS:")
print(y.value_counts())


In [4]:
df.shape

(527, 24)

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Các đặc trưng dạng số thực và phân loại dùng cho pipeline
numeric_features = ["wind_speed", "battery_remaining", "actual_carry_weight", "altitude", "distance_flown", "gps_accuracy"]
categorical_features = ["payload_type", "obstacles_encountered"]

# Thiết lập đường ống xử lý tự động cho từng nhóm đặc trưng
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Gom hai nhánh xử lý vào bộ tiền xử lý trung tâm
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("✔ Khởi tạo Pipeline tiền xử lý dữ liệu tự động thành công!")


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib

# Thiết lập mô hình đối chiếu 1: Logistic Regression với class_weight balanced
log_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])

# Huấn luyện mô hình
log_model.fit(X_train, y_train)

# Dự báo và đánh giá định lượng
y_pred_log = log_model.predict(X_test)

print("=== KẾT QUẢ MÔ HÌNH LOGISTIC REGRESSION ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_log):.4f}")
print("\nMa trận nhầm lẫn (Confusion Matrix):")
print(confusion_matrix(y_test, y_pred_log))
print("\nBáo cáo phân loại chi tiết (Classification Report):")
print(classification_report(y_test, y_pred_log))

# Lưu model pkl
joblib.dump(log_model, "../models/logistic_pipeline.pkl")
print("✔ Đã lưu mô hình Logistic Regression thành công!")


In [7]:
from sklearn.ensemble import RandomForestClassifier
import joblib

# Thiết lập mô hình đối chiếu 2: Random Forest Classifier với 200 cây quyết định
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"))
])

# Huấn luyện mô hình
rf_model.fit(X_train, y_train)

# Dự báo và đánh giá định lượng
y_pred_rf = rf_model.predict(X_test)

print("=== KẾT QUẢ MÔ HÌNH RANDOM FOREST ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print("\nMa trận nhầm lẫn (Confusion Matrix):")
print(confusion_matrix(y_test, y_pred_rf))
print("\nBáo cáo phân loại chi tiết (Classification Report):")
print(classification_report(y_test, y_pred_rf))

# Lưu model pkl
joblib.dump(rf_model, "../models/random_forest_pipeline.pkl")
print("✔ Đã lưu mô hình Random Forest thành công!")


,application,drone_size,drone_model,manufacturer,propeller_count,max_carry_weight,actual_carry_weight,payload_type,payload_description,altitude,flight_duration,distance_flown,battery_remaining,gps_accuracy,wind_speed,obstacles_encountered,overweight_flag,weight_ratio,risk_score
0,Package Delivery,Medium,FlyHigh 300,AeroCorp,4,5.0,2.5,Package,Small consumer goods,50,25.0,8.0,85,2.1,3.5,No,0,0.500000,0.897508
1,Infrastructure Inspection,Large,Inspecta X,SkyView Inc.,6,15.0,8.0,"Camera, Sensor","High-resolution camera, thermal sensor",120,45.0,15.0,68,1.8,6.2,Yes,0,0.533333,1.356176
2,Agricultural Spraying,Large,CropMaster,AgriDrones,8,20.0,18.0,Liquid Tank,Pesticide solution,30,30.0,10.0,40,3.5,7.8,Yes,0,0.900000,1.490365
3,Aerial Photography,Small,SnapShot Mini,PhotoFly,4,1.0,0.5,Camera,Lightweight action camera,80,20.0,5.0,92,1.2,2.1,No,0,0.500000,1.076124
4,Surveillance,Medium,Watcher Pro,SecureTech,4,7.0,3.0,"Camera, Sensor","Infrared camera, motion detection sensor",100,60.0,12.0,55,2.5,4.9,No,0,0.428571,1.229845


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((421, 19), (106, 19), (421,), (106,))

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=200, random_state=42))
])

model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['propeller_count',
                                                   'max_carry_weight',
                                                   'actual_carry_weight',
                                                   'altitude',
                                                   'flight_duration',
                                                   'distance_flown',
                                                   'battery_remaining',
                                                   'gps_accuracy', 'wind_spe

In [10]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9905660377358491
[[  0   1]
 [  0 105]]
              precision    recall  f1-score   support

     Aborted       0.00      0.00      0.00         1
   Completed       0.99      1.00      1.00       105

    accuracy                           0.99       106
   macro avg       0.50      0.50      0.50       106
weighted avg       0.98      0.99      0.99       106



/opt/conda/envs/anaconda-2022.05-py39/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/envs/anaconda-2022.05-py39/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/envs/anaconda-2022.05-py39/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start,

In [11]:
from sklearn.linear_model import LogisticRegression

balanced_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

balanced_model.fit(X_train, y_train)

y_pred2 = balanced_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred2))
print(confusion_matrix(y_test, y_pred2))
print(classification_report(y_test, y_pred2))

Accuracy: 9.905660377358491e-01
[[  0   0   1]
 [  0 105   0]
 [  0   0   0]]
                     precision    recall  f1-score   support

            Aborted       0.00      0.00      0.00         1
          Completed       1.00      1.00      1.00       105
Landed Unexpectedly       0.00      0.00      0.00         0

           accuracy                           0.99       106
          macro avg       0.33      0.33      0.33       106
       weighted avg       0.99      0.99      0.99       106



/opt/conda/envs/anaconda-2022.05-py39/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/envs/anaconda-2022.05-py39/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/envs/anaconda-2022.05-py39/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(res

In [12]:
y.value_counts()

Completed              522
Aborted                  3
Landed Unexpectedly      2
Name: flight_status, dtype: int64

In [13]:
y_binary = y.replace({
    "Completed": "Completed"
})

In [14]:
y_binary = y.apply(lambda x: "Completed" if x == "Completed" else "Non-completed")

In [15]:
y_binary = y.apply(lambda x: "Completed" if x == "Completed" else "Non-completed")
y_binary.value_counts()

Completed        522
Non-completed      5
Name: flight_status, dtype: int64

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

balanced_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

balanced_model.fit(X_train, y_train)

y_pred = balanced_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 1.0
[[105   0]
 [  0   1]]
               precision    recall  f1-score   support

    Completed       1.00      1.00      1.00       105
Non-completed       1.00      1.00      1.00         1

     accuracy                           1.00       106
    macro avg       1.00      1.00      1.00       106
 weighted avg       1.00      1.00      1.00       106



In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

log_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

log_model.fit(X_train, y_train)
y_pred_log = log_model.predict(X_test)

print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print(confusion_matrix(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))

Logistic Regression
Accuracy: 1.0
[[105   0]
 [  0   1]]
               precision    recall  f1-score   support

    Completed       1.00      1.00      1.00       105
Non-completed       1.00      1.00      1.00         1

     accuracy                           1.00       106
    macro avg       1.00      1.00      1.00       106
 weighted avg       1.00      1.00      1.00       106



In [19]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"))
])

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("Random Forest")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Random Forest
Accuracy: 0.9905660377358491
[[105   0]
 [  1   0]]
               precision    recall  f1-score   support

    Completed       0.99      1.00      1.00       105
Non-completed       0.00      0.00      0.00         1

     accuracy                           0.99       106
    macro avg       0.50      0.50      0.50       106
 weighted avg       0.98      0.99      0.99       106



/opt/conda/envs/anaconda-2022.05-py39/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/envs/anaconda-2022.05-py39/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/envs/anaconda-2022.05-py39/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start,

In [20]:
import pandas as pd

hyperparams = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "max_iter": 1000,
        "class_weight": "balanced",
        "random_state": None,
        "n_estimators": None
    },
    {
        "model": "Random Forest",
        "max_iter": None,
        "class_weight": "balanced",
        "random_state": 42,
        "n_estimators": 200
    }
])

hyperparams

,model,max_iter,class_weight,random_state,n_estimators
0,Logistic Regression,1000.0,balanced,NaN,NaN
1,Random Forest,NaN,balanced,42.0,200.0


In [21]:
hyperparams.to_csv("hyperparameters_log.csv", index=False)